# 🧾 Procesador y Aumentador de Dataset de Facturas

Este notebook procesa facturas y sus anotaciones JSON:

1. **Renombra** facturas y JSONs de forma correlativa
2. **Genera 16 variaciones** mediante desplazamientos en diferentes direcciones
3. **Mantiene** la data extraída en cada JSON

---

## 📊 **Iteraciones por factura: 16**

**8 desplazamientos completos (±10px):**
- Derecha, Izquierda, Abajo, Arriba
- 4 diagonales

**8 desplazamientos medios (±5px):**
- Las mismas 8 direcciones con la mitad de desplazamiento

**Resultado:** 10 facturas → 160 aumentadas = **170 total**

---

## 🔗 Paso 0: Clonar Código desde GitHub (OPCIONAL)

Si quieres usar el código directamente desde Git, ejecuta esta celda.

**Alternativa:** Puedes saltar este paso, el código está incluido en este notebook.

In [ ]:
# Clonar repositorio (OPCIONAL - solo si quieres el código desde Git)
!git clone https://github.com/GynoRomeroPrado/Creador-De-Factura.git
%cd Creador-De-Factura

# Cambiar al branch correcto
!git checkout claude/colab-invoice-processor-011CUuowDXaNFjHtgp9gChim

print("\n✅ Repositorio clonado correctamente")
print("\n📁 Archivos disponibles:")
!ls -la *.py *.ipynb *.md

## 📦 Paso 1: Instalación de Dependencias

In [ ]:
print("🔧 Instalando dependencias...\n")

# Instalar librerías necesarias
!pip install -q pdf2image pillow
!apt-get install -q poppler-utils

print("\n✅ Dependencias instaladas correctamente")

## 💾 Paso 2: Montar Google Drive

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive montado correctamente")

## ⚙️ Paso 3: Configuración

**⚠️ IMPORTANTE:** Modifica las rutas según tu estructura de Drive

In [ ]:
# ========================================
# CONFIGURACIÓN - AJUSTA ESTAS RUTAS
# ========================================

# Carpeta de entrada con tus facturas y JSONs
INPUT_FOLDER = "/content/drive/MyDrive/Facturas"  # ⚠️ CAMBIAR AQUÍ

# Carpeta de salida para facturas procesadas
OUTPUT_FOLDER = "/content/drive/MyDrive/Facturas_Procesadas"  # ⚠️ CAMBIAR AQUÍ

# Rango de píxeles para desplazamientos (10px recomendado)
PIXEL_RANGE = 10

# ========================================
# ITERACIONES POR FACTURA
# ========================================
# Total: 16 variaciones por factura
#   - 8 desplazamientos completos (±10px)
#   - 8 desplazamientos medios (±5px)
# ========================================

print("="*50)
print("⚙️  CONFIGURACIÓN DEL PROCESAMIENTO")
print("="*50)
print(f"\n📁 Carpeta de entrada: {INPUT_FOLDER}")
print(f"📁 Carpeta de salida: {OUTPUT_FOLDER}")
print(f"📏 Rango de píxeles: ±{PIXEL_RANGE}px")
print(f"🔄 Iteraciones por factura: 16")
print(f"\n💡 Ejemplo: 10 facturas → 160 aumentadas = 170 total")

# Verificar que la carpeta existe
if os.path.exists(INPUT_FOLDER):
    files = os.listdir(INPUT_FOLDER)
    images = [f for f in files if f.endswith(('.jpg', '.jpeg', '.png', '.pdf', '.JPG', '.JPEG', '.PNG', '.PDF'))]
    jsons = [f for f in files if f.endswith('.json')]
    print(f"\n✅ Carpeta encontrada")
    print(f"   📄 {len(images)} imágenes/PDFs")
    print(f"   📋 {len(jsons)} archivos JSON")
    if len(images) > 0:
        estimated = len(images) * 16
        print(f"\n📊 Facturas estimadas a generar: {estimated}")
        print(f"📊 Total estimado en dataset: {len(images) + estimated}")
else:
    print(f"\n❌ ERROR: La carpeta {INPUT_FOLDER} no existe")
    print("Por favor, ajusta INPUT_FOLDER con la ruta correcta")

## 🔨 Paso 4: Cargar Procesador

Esta celda define toda la lógica del procesamiento

In [ ]:
import json
import shutil
from pathlib import Path
from typing import List, Tuple, Dict
from PIL import Image
from pdf2image import convert_from_path


class InvoiceDatasetProcessor:
    """Clase para procesar y aumentar el dataset de facturas"""

    def __init__(self, drive_folder_path: str, output_folder_path: str = None):
        self.input_folder = Path(drive_folder_path)
        self.output_folder = Path(output_folder_path) if output_folder_path else self.input_folder

        # Crear subcarpetas organizadas
        self.organized_folder = self.output_folder / "organized"
        self.augmented_folder = self.output_folder / "augmented"

        self.organized_folder.mkdir(parents=True, exist_ok=True)
        self.augmented_folder.mkdir(parents=True, exist_ok=True)

    def get_invoice_pairs(self) -> List[Tuple[Path, Path]]:
        """Encuentra pares de facturas y sus JSON correspondientes"""
        pairs = []
        invoice_extensions = ['.jpg', '.jpeg', '.png', '.pdf']
        invoice_files = []

        for ext in invoice_extensions:
            invoice_files.extend(self.input_folder.glob(f'*{ext}'))
            invoice_files.extend(self.input_folder.glob(f'*{ext.upper()}'))

        for invoice_file in invoice_files:
            json_file = invoice_file.with_suffix('.json')
            if json_file.exists():
                pairs.append((invoice_file, json_file))
            else:
                print(f"⚠️  Advertencia: No se encontró JSON para {invoice_file.name}")

        return pairs

    def rename_and_organize(self) -> List[Tuple[Path, Path]]:
        """Renombra las facturas y JSONs de forma correlativa"""
        print("\n📋 Paso 1: Renombrando y organizando facturas...\n")

        pairs = self.get_invoice_pairs()
        organized_pairs = []

        for idx, (invoice_file, json_file) in enumerate(pairs, start=1):
            extension = invoice_file.suffix
            new_invoice_name = f"factura_{idx:04d}{extension}"
            new_json_name = f"factura_{idx:04d}.json"

            new_invoice_path = self.organized_folder / new_invoice_name
            new_json_path = self.organized_folder / new_json_name

            shutil.copy2(invoice_file, new_invoice_path)

            with open(json_file, 'r', encoding='utf-8') as f:
                json_data = json.load(f)

            # Mantener toda la data extraída y agregar metadata
            json_data['archivo_factura'] = new_invoice_name
            json_data['id_correlativo'] = f"factura_{idx:04d}"

            with open(new_json_path, 'w', encoding='utf-8') as f:
                json.dump(json_data, f, ensure_ascii=False, indent=2)

            organized_pairs.append((new_invoice_path, new_json_path))
            print(f"  ✓ {invoice_file.name} → {new_invoice_name}")

        print(f"\n✅ {len(organized_pairs)} pares de facturas organizados")
        return organized_pairs

    def convert_pdf_to_image(self, pdf_path: Path) -> Image.Image:
        """Convierte la primera página de un PDF a imagen"""
        images = convert_from_path(str(pdf_path), first_page=1, last_page=1, dpi=200)
        return images[0]

    def load_invoice_image(self, invoice_path: Path) -> Image.Image:
        """Carga una factura como imagen (PDF o imagen)"""
        if invoice_path.suffix.lower() == '.pdf':
            return self.convert_pdf_to_image(invoice_path)
        else:
            return Image.open(invoice_path)

    def apply_shift(self, image: Image.Image, shift_x: int, shift_y: int,
                    fill_color: tuple = (255, 255, 255)) -> Image.Image:
        """Aplica un desplazamiento a la imagen"""
        if image.mode != 'RGB':
            image = image.convert('RGB')

        width, height = image.size
        shifted_image = Image.new('RGB', (width, height), fill_color)
        shifted_image.paste(image, (shift_x, shift_y))

        return shifted_image

    def get_augmentation_configs(self, pixel_range: int = 10) -> List[Dict]:
        """Genera configuraciones de aumento de datos - 16 iteraciones"""
        configs = []

        # 8 direcciones principales (±pixel_range)
        directions = [
            ("derecha", pixel_range, 0),
            ("izquierda", -pixel_range, 0),
            ("abajo", 0, pixel_range),
            ("arriba", 0, -pixel_range),
            ("diagonal_superior_derecha", pixel_range, -pixel_range),
            ("diagonal_superior_izquierda", -pixel_range, -pixel_range),
            ("diagonal_inferior_derecha", pixel_range, pixel_range),
            ("diagonal_inferior_izquierda", -pixel_range, pixel_range),
        ]

        for name, shift_x, shift_y in directions:
            configs.append({
                'name': name,
                'shift_x': shift_x,
                'shift_y': shift_y
            })

        # 8 desplazamientos intermedios (±pixel_range/2)
        for name, shift_x, shift_y in directions:
            configs.append({
                'name': f"{name}_medio",
                'shift_x': shift_x // 2 if shift_x != 0 else 0,
                'shift_y': shift_y // 2 if shift_y != 0 else 0
            })

        return configs  # Total: 16 configuraciones

    def augment_dataset(self, organized_pairs: List[Tuple[Path, Path]],
                       pixel_range: int = 10) -> int:
        """Aumenta el dataset aplicando transformaciones - 16 por factura"""
        print(f"\n🔄 Paso 2: Aumentando dataset con desplazamientos de ±{pixel_range}px...\n")

        augmentation_configs = self.get_augmentation_configs(pixel_range)
        print(f"📊 Total de variaciones por factura: {len(augmentation_configs)}\n")
        
        total_generated = 0

        for invoice_path, json_path in organized_pairs:
            with open(json_path, 'r', encoding='utf-8') as f:
                original_json = json.load(f)

            try:
                original_image = self.load_invoice_image(invoice_path)
                base_name = invoice_path.stem

                print(f"  Procesando: {invoice_path.name}")

                for idx, config in enumerate(augmentation_configs, start=1):
                    augmented_image = self.apply_shift(
                        original_image,
                        config['shift_x'],
                        config['shift_y']
                    )

                    aug_name = f"{base_name}_aug_{idx:02d}_{config['name']}"
                    aug_image_path = self.augmented_folder / f"{aug_name}.png"
                    augmented_image.save(aug_image_path, 'PNG', quality=95)

                    # Copiar TODA la data extraída original
                    aug_json = original_json.copy()
                    aug_json['archivo_factura'] = f"{aug_name}.png"
                    aug_json['id_correlativo'] = aug_name
                    
                    # Metadata de augmentation (opcional, puedes removerlo si no lo necesitas)
                    aug_json['augmentation'] = {
                        'original_file': invoice_path.name,
                        'transformation': config['name'],
                        'shift_x': config['shift_x'],
                        'shift_y': config['shift_y']
                    }

                    aug_json_path = self.augmented_folder / f"{aug_name}.json"
                    with open(aug_json_path, 'w', encoding='utf-8') as f:
                        json.dump(aug_json, f, ensure_ascii=False, indent=2)

                    total_generated += 1

                    if idx % 8 == 0:
                        print(f"    ✓ {idx}/{len(augmentation_configs)} variaciones generadas...")

                print(f"  ✅ Completado: {len(augmentation_configs)} variaciones\n")

            except Exception as e:
                print(f"  ❌ Error procesando {invoice_path.name}: {str(e)}\n")
                continue

        return total_generated

    def generate_dataset_report(self) -> Dict:
        """Genera un reporte del dataset procesado"""
        organized_files = list(self.organized_folder.glob('factura_*.json'))
        augmented_files = list(self.augmented_folder.glob('*.json'))

        report = {
            'facturas_originales_organizadas': len(organized_files),
            'facturas_aumentadas_generadas': len(augmented_files),
            'total_facturas_dataset': len(organized_files) + len(augmented_files),
            'variaciones_por_factura': 16,
            'carpeta_organizadas': str(self.organized_folder),
            'carpeta_aumentadas': str(self.augmented_folder)
        }

        report_path = self.output_folder / 'dataset_report.json'
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, ensure_ascii=False, indent=2)

        return report


print("✅ Procesador cargado correctamente")
print("📊 Configurado para generar 16 variaciones por factura")

## 🚀 Paso 5: Ejecutar Procesamiento

Esta celda ejecuta todo el proceso de aumento de datos

In [ ]:
print("=" * 70)
print("🚀 PROCESADOR DE DATASET DE FACTURAS")
print("=" * 70)

# Crear procesador
processor = InvoiceDatasetProcessor(INPUT_FOLDER, OUTPUT_FOLDER)

# Paso 1: Renombrar y organizar
organized_pairs = processor.rename_and_organize()

if not organized_pairs:
    print("\n❌ No se encontraron pares de facturas y JSONs")
else:
    # Paso 2: Aumentar dataset (16 variaciones por factura)
    total_generated = processor.augment_dataset(organized_pairs, PIXEL_RANGE)

    # Generar reporte
    print("\n" + "=" * 70)
    print("📊 REPORTE FINAL")
    print("=" * 70)

    report = processor.generate_dataset_report()

    print(f"\n✓ Facturas originales organizadas: {report['facturas_originales_organizadas']}")
    print(f"✓ Facturas aumentadas generadas: {report['facturas_aumentadas_generadas']}")
    print(f"✓ Variaciones por factura: {report['variaciones_por_factura']}")
    print(f"✓ Total facturas en dataset: {report['total_facturas_dataset']}")
    print(f"\n📁 Carpeta organizadas: {report['carpeta_organizadas']}")
    print(f"📁 Carpeta aumentadas: {report['carpeta_aumentadas']}")
    print(f"\n🎉 Proceso completado exitosamente!")

## 📊 Paso 6: Visualizar Resultados (Opcional)

Muestra algunas de las facturas aumentadas generadas

In [ ]:
import matplotlib.pyplot as plt

# Obtener algunas imágenes generadas
augmented_images = list(processor.augmented_folder.glob('*.png'))[:9]

if augmented_images:
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.flatten()

    for idx, img_path in enumerate(augmented_images):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.stem, fontsize=8)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("No hay imágenes aumentadas para visualizar")

## 📥 Paso 7: Descargar Reporte (Opcional)

In [ ]:
from google.colab import files

# Descargar el reporte JSON
report_path = str(processor.output_folder / 'dataset_report.json')

if os.path.exists(report_path):
    files.download(report_path)
    print("✅ Reporte descargado")
else:
    print("❌ No se encontró el reporte")